# Phân tích RQ2–RQ4

Hồ Nhật Triều (20236003)

Notebook này **chỉ gọi hàm và hiển thị**. Mọi phép tính nằm trong `src/analysis/`.

Giải thích từng bước: `docs/analysis/rq2_lam_tay.md`, `rq3_lam_tay.md`, `rq4_lam_tay.md`

## Nạp dữ liệu

Hiện dùng dữ liệu giả. Khi bước bóc skill giao file thật, bỏ chú thích ở ô dưới
và xóa phần dữ liệu giả — ba hàm `run_rq*` không phải sửa gì.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from src.analysis import loaders, plots, rq2, rq3, rq4, synthetic

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 50)

In [ ]:
# --- dữ liệu giả (hiện tại) ---
df = synthetic.generate_jobs(n=400)
M = synthetic.build_matrix(df)
titles = df.set_index('job_id')['normalized_job_title']

# --- dữ liệu thật (khi có) ---
# M = loaders.load_skill_matrix('data/features/job_skill_matrix.parquet')
# for canh_bao in loaders.warn_about(M):
#     print('CẢNH BÁO:', canh_bao)
# titles = pd.read_parquet('data/processed/jobs.parquet').set_index('job_id')['normalized_job_title']

print(f'{M.shape[0]} tin x {M.shape[1]} skill')
M.head(3)

---
# RQ2 — Thị trường cần skill gì

Hai câu hỏi: skill nào phổ biến, và skill nào đi cùng nhau.

In [ ]:
r2 = rq2.run_rq2(M)
r2['skills_per_posting']

### Skill phổ biến nhất

In [ ]:
r2['skill_frequency'].head(15)

### Cặp skill liên quan mạnh nhất

`joint` là số đếm thô. `npmi` mới cho biết cặp đó có thật sự liên quan,
hay chỉ gặp nhau nhiều vì cả hai đều phổ biến.

In [ ]:
r2['pairs'].head(10)

### Cái bẫy

Python và SQL cùng xuất hiện rất nhiều lần, nhưng npmi âm — chúng gặp nhau
chỉ vì cả hai đều ở khắp nơi.

In [ ]:
p = r2['pairs']
p[(p.skill_a.isin(['Python','SQL'])) & (p.skill_b.isin(['Python','SQL']))]

### Nhóm skill tự hình thành

In [ ]:
(pd.Series(r2['communities'], name='community')
   .rename_axis('skill').reset_index()
   .groupby('community')['skill'].apply(lambda s: ', '.join(sorted(s))))

---
# RQ3 — Tên công việc có khớp skill không

Hai chiều, làm riêng.

In [ ]:
r3 = rq3.run_rq3(M, titles)
r3['status']

### Chiều 1 — cùng title, skill có giống nhau không

`dispersion` cao nghĩa là cái tên nói lên ít.

Con số đó vô nghĩa nếu đứng một mình, nên có `baseline_p5` — mức của nhóm
bốc ngẫu nhiên cùng cỡ. `informative=True` nghĩa là title thấp hơn hẳn mức
ngẫu nhiên, tức cái tên có mang thông tin.

In [ ]:
r3['dispersion']

### Chiều 2 — title khác nhau, skill có giống nhau không

In [ ]:
r3['title_similarity'].head(10)

### Giống nhau ở chỗ nào, khác nhau ở chỗ nào

Cột skill phân biệt **rỗng** nghĩa là hai cái tên đòi y hệt nhau —
hai tên, một công việc.

In [ ]:
r3['notable']

---
# RQ4 — Giấu title đi, dữ liệu tự chia thành mấy nhóm

Câu chính của đồ án.

Thứ tự bắt buộc: gom nhóm chỉ bằng skill → chọn k → chốt cụm → **rồi mới**
ghép title vào.

In [ ]:
r4 = rq4.run_rq4(M, titles=titles)
print(r4['message'])

### Quét k

Ba chỉ số cho từng số cụm. Stability được xếp trên hai cái kia: một cách chia
có thể trông gọn nhưng biến mất khi đổi mẫu.

In [ ]:
r4['sweep']

In [ ]:
print('Chọn k =', r4['chosen_k'])

### Mỗi cụm gồm skill gì — title vẫn đang bị giấu

In [ ]:
r4['profiles']

### Giờ mới ghép title vào

In [ ]:
r4['profiles'].merge(r4['unblinded'], on='cluster')

### Mức khớp giữa cụm và title

Gần 1: tên nghề phản ánh đúng công việc. Gần 0: hai thứ đã lệch nhau.

In [ ]:
r4['agreement']

### Hình

Hai khung PCA cạnh nhau. Giống nhau thì tên nghề khớp công việc; khác nhau
thì đã lệch. Chênh lệch giữa hai khung chính là câu trả lời của RQ4.

In [ ]:
from IPython.display import Image

plots.ve_pca(r4, titles, '../reports/analysis/rq4_pca.png')
Image('../reports/analysis/rq4_pca.png')

In [ ]:
plots.ve_quet_k(r4['sweep'], r4['chosen_k'], '../reports/analysis/rq4_sweep_k.png')
Image('../reports/analysis/rq4_sweep_k.png')

---
## Xuất toàn bộ kết quả ra file

Chạy ngoài terminal, ghi mọi bảng và hình vào `reports/analysis/`:

```
python scripts/run_analysis.py
```